# 🎯 Squeeze breakout — intraday backtest & optimize

**The idea.** Most intraday channel breakouts fail (practitioner datasets put raw false-break rates at 50–70%). `donchian_breakout` trades them all anyway; this strategy is its **selective complement**: a break of the `break_window`-minute channel is taken only when **every enabled gate agrees** — the tape was *compressed* before the move started (lagged `ATR / SMA(ATR, squeeze_window)` below `squeeze_thresh`; volatility clusters, expansion follows compression), *participation* is abnormal (trailing `vol_bars`-minute volume at least `vol_mult`× its time-of-day expectation), and the break goes *with* the session's trend (`SMA(trend_window)`). Few trades, confluent evidence for each. Exits are a layered bracket scaled by **normal** volatility (`max(ATR, its baseline)`, never the squeeze-compressed reading): fixed stop at `sl_mult` scales, optional target at `tp_mult` (0 = winners run), a `horizon`-minute time stop, and `eod_exit` (default) goes flat on each day's final bar. Deliberately **no trailing stop** — the trail is donchian's exit thesis, the bracket is this one's. Full thesis in [README.md](./README.md).

**The universe.** It trades `HighOctaneTop20`: the sibling repo's **HighOctane** universe (top volatility tercile within sector, imported with `tsim import-universes`), narrowed to the **20 most liquid names by trailing-90-day average dollar volume** (`tsim subset --universe HighOctane --top 20`) — compression→expansion cycles need volatility, 1-minute fills and per-leg costs need liquidity, and the pinned list in `config/universes.yaml` keeps runs reproducible.

**Train vs holdout.** The `data:` block sets `end` to a deliberate **train cutoff** — the sweep and the optimizer only ever see data up to it. §12 replays the picked parameters on the untouched ~2-month tail, the one number in this notebook the optimizer could not have fit. Every knob lives in [`config.yaml`](./config.yaml) (defaults, sweep grid, optimization space) — this notebook only *drives* the shared engine: **load → overview → backtest → inspect → sweep → optimize → boundary-check → verify → ablate & stress → out-of-sample**. The same steps work for every strategy in this repo.


## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import backtest as bt, jobs, optimize as opt, strategies
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple import plots

strat = strategies.load("squeeze_breakout")
print("data :", strat.data)
print("params:", strat.params)


In [ ]:
d = strat.data
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]),
                         d["bar"], d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
print(f"panel: {panel.close.shape[0]:,} bars x {len(panel.symbols)} symbols "
      f"({panel.close.index[0]:%Y-%m-%d} → {panel.close.index[-1]:%Y-%m-%d})")

## 2 · Strategy overview — hyperparameters and search ranges

What each knob does, before any number crunching:

| param | role in the strategy | intuition |
|---|---|---|
| `squeeze_thresh` | max lagged ATR/baseline ratio | **the** selectivity knob: lower = quieter tape required (median ratio ≈ 0.82 on this panel); 0 = gate off (ablation) |
| `squeeze_lag` | bars the squeeze ratio is lagged | "quiet before the move *started*" — doesn't veto multi-bar breaks |
| `break_window` | channel lookback (bars = minutes on 1m) | what counts as a new extreme; donchian's knob |
| `vol_mult` | participation multiple | how abnormal the last `vol_bars` minutes' volume must be; 0 = gate off |
| `vol_bars` | bars in the volume sums | k-bar form rewards building participation, not one print |
| `trend_window` | regime SMA (bars) | break only with the drift; ≥ 2× `break_window`; 0 = gate off |
| `atr_window` / `squeeze_window` | ATR + its baseline (bars) | the compression ratio's two legs; the baseline also floors the stop scale |
| `sl_mult` | stop distance (stop-scale units) | caps the loss; < 1 is inside 1m noise, searched from 1.0 |
| `tp_mult` | target distance; 0 = none | 0 rides winners to the time stop; > 0 books an explicit R:R |
| `horizon` | time stop (bars held) | dead breaks are dead risk; momentum decays fast |
| `direction` | long / short / both | gates are symmetric; default both |
| `skip_open` / `skip_close` | entry blackout (minutes) | no fills into the auction chaos; inert on 1d |
| `eod_exit` | flat at session close | stops cannot price overnight gaps; stays on |

The table below shows, per parameter: the **default** used by the single
backtest (§3), the hand-picked **sweep grid** (§5), and the **Optuna search
range** (§6). A `—` means the stage keeps the default.


In [ ]:
import pandas as pd

def _fmt_space(spec):
    """One readable line for an optuna.space entry."""
    if spec["type"] == "categorical":
        return " | ".join(map(str, spec["choices"]))
    tail = " (log)" if spec.get("log") else (f", step {spec['step']}" if spec.get("step") else "")
    return f"{spec['low']} – {spec['high']}{tail}"

pd.DataFrame({
    "default": pd.Series(strat.params),
    "sweep grid (§5)": pd.Series({k: ", ".join(map(str, v)) for k, v in strat.sweep.items()}),
    "optuna search range (§6)": pd.Series({k: _fmt_space(s)
                                           for k, s in strat.optuna["space"].items()}),
}).fillna("—")

## 3 · Single backtest with the config defaults

Engine rules (identical for every strategy): signals execute at the **next
bar's adjusted open**, realistic costs are always charged, every symbol trades
its own starting cash.

> **Capital model:** ONE shared \$10k wallet (engine cash-shared mode) — each entry deploys `cash_frac` of the *free* cash, so the book is never more than 100% invested. The repo-wide comparability rule; see `notebooks/04_compare.ipynb` for the cross-strategy scoreboard.

In [ ]:
import pandas as pd
pf = bt.run_strategy(strat, panel, settings=settings)
stats = bt.portfolio_stats(pf)
pd.Series(stats).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity, config defaults").show()

## 4 · Inspect a symbol — trades and gate overlays

`inspect_symbol` is the microscope of this notebook, used twice: here on the
most-traded symbol with the *default* parameters, and again in §10 on the
*optimized* ones. How to read the figure:

* **Price panel** — the `break_window`-bar channel bounds (drawn from close as
  a proxy; signals use true high/low) plus the trend SMA. ▲ blue = long entry
  on a confluence-gated upside break, ▼ orange = short entry on a downside
  break, ✕ = exit (stop, target, `horizon` time stop or session close); the
  entry→exit segment is green when the trade won, red when it lost.
* **Momentum panel** — the lagged squeeze ratio vs its threshold: *below* the
  line = compressed tape, where the squeeze gate allows entries.
* **Volume panel** — the time-of-day expected volume with the `vol_mult`×
  trigger line; marked bars passed the participation gate.


In [ ]:
def inspect_symbol(sym, pf, params):
    """Deep-dive one symbol of a portfolio: annotated chart + its trade table."""
    all_trades = bt.trade_records(pf)
    merged = {**strat.params, **params, "bar": panel.bar, "tz": panel.tz}
    overlays = (strat.indicators(panel.close[sym], panel.volume[sym], **merged)
                if strat.indicators else None)
    plots.plot_symbol(panel.close[sym], panel.volume[sym], all_trades, symbol=sym,
                      overlays=overlays, equity=pf.value(group_by=False)[sym]).show()
    mine = all_trades[all_trades["symbol"] == sym]
    return mine.sort_values("entry_ts").reset_index(drop=True).round(3)

trades = bt.trade_records(pf)
busiest = trades["symbol"].value_counts().index[0]
inspect_symbol(busiest, pf, strat.params)

## 5 · Grid sweep — a coarse map of the parameter space

Before optimizing, sweep the small hand-picked grid from `config.yaml`. The
ranked bars show the best single combos; the **heatmap grid** below them is the
multi-objective view — the same parameter pair judged by Sharpe, total return
*and* breadth (`pct_profitable` = share of symbols that made money) at once, so
a region that pays on return but is carried by one name stands out immediately.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
print(f"{len(grid)} combos")
sweep_stats = bt.sweep(strat, panel, grid, settings=settings)
sweep_stats.sort_values("sharpe", ascending=False).head(8).round(3)

In [ ]:
plots.plot_sweep(sweep_stats).show()
plots.plot_heatmap_grid(
    sweep_stats,
    pairs=[("squeeze_thresh", "vol_mult"), ("break_window", "tp_mult")],
    metrics=("sharpe", "total_return", "pct_profitable"),
).show()


## 6 · Optimize (resumable, runs outside the notebook)

`jobs.run_optimize` hands the work to a `tsim optimize` subprocess. The study
maximizes **Sharpe and total return together**, so the result is a *Pareto
front* — the set of parameter sets where improving one objective must hurt the
other. Every finished trial is checkpointed to `studies/{study}.log`;
**re-running this cell resumes** exactly where it stopped (Ctrl-C / kernel
restarts lose nothing).

While it runs, every finished trial streams one line below — its objectives,
the current front size, and its parameters — so you can watch the search move.
For a richer live view, copy the **`uvx optuna-dashboard …`** command from the
first output line into a second terminal: it serves parameter importances,
objective history and the front at http://127.0.0.1:8080 while the study runs.

In [ ]:
N_TRIALS = 1000   # raise for a deeper search; re-runs accumulate into the same study
jobs.run_optimize("squeeze_breakout", n_trials=N_TRIALS)


## 7 · Read the study: Pareto front, admissibility, the picked trial

Gray dots are all completed trials; the blue **staircase** is the exact
boundary of the dominated region; the orange star is the picked trial. Hover
any point for its full parameter set.

**Admissibility first.** The Sharpe objective is a per-symbol MEAN in which
symbols that never trade silently vanish — and a four-gate strategy owns a
selectivity dial, so the front's Sharpe end attracts few-trade, few-symbol
artifacts (three lucky fills annualize to a huge Sharpe). Only front rows
with at least `MIN_TRADES` train trades are eligible for picking, and the
weights lean on total return, which silent symbols *do* dilute.


> **Note:** new trials are floor-pruned in-objective via `optuna.min_trades` in `config.yaml`; the admissibility filter below still guards older trials in resumed journals. The study name was bumped when the capital model changed to the shared wallet — earlier journals scored independent per-symbol books and are not comparable.

In [ ]:
import pandas as pd

study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)

# Objectives are benchmark-relative and annualized: info_ratio (active return per unit of
# tracking risk) and excess_cagr (annualized edge over equal-weight buy-and-hold). Two
# guards before picking: (1) an admissibility floor on train trades — a handful of lucky
# fills can post a huge info_ratio that means nothing; (2) `opt.pick_from_front`, the ONE
# picking rule in the repo (opt.PICK_WEIGHTS, balanced), which collapses tied outcomes
# first so a coin flip between identical-scoring trials cannot masquerade as a decision.
# Watch `n_tied`: > 1 means the pick was arbitrary among that many equal strategies.
MIN_TRADES = 300      # admissibility floor for TRAIN trades — below this, ranking is noise.
                      # On a small admissible front min-max scoring is knife-edged — sanity-
                      # check the runner-up rows in §12's consistency replay before believing it

eligible = front[front["n_trades"] >= MIN_TRADES]
if eligible.empty:
    print(f"!! no Pareto trial reaches {MIN_TRADES} train trades — showing the raw front; "
          "treat any pick as inadmissible")
    eligible = front
best_row = opt.pick_from_front(eligible)
best = next(t for t in study.best_trials if t.number == int(best_row["trial"]))
print(f"front: {len(front)} trials, {len(eligible)} admissible (>= {MIN_TRADES} trades), "
      f"{int(best_row['front_distinct'])} distinct outcomes; picked trial {best.number}"
      + (f" — TIED with {int(best_row['n_tied']) - 1} other(s), so this pick is arbitrary"
         if best_row["n_tied"] > 1 else ""))
plots.plot_pareto(front, everything, picked=best.number).show()
eligible.head(8).round(3)


## 8 · Boundary check — did the optimum hit the search-space edge?

A picked trial sitting **on the edge of a searched range** is a hint that the
true optimum may lie outside it. `optimize.boundary_report` compares the
picked parameters (and the whole Pareto front) with `search:` ranges from
`config.yaml`: any numeric parameter within 5% of a bound — measured on the
searched scale, so `log: true` ranges are judged in log space — is flagged
with the bound to widen. Categorical parameters have no edges and are skipped.

If something is flagged: widen that bound in `config.yaml`, re-run §6 (the
same study name resumes, and the sampler explores the new region; rename the
study to restart clean), then re-read §7–§8 until the front sits inside the
box. `tsim optimize` prints the same check at the end of every CLI run.

In [ ]:
report = opt.boundary_report(best.params, strat.optuna["space"], front=front)
flagged = report[report["action"] != "ok"]
print("all clear — the searched ranges bracket the optimum" if flagged.empty
      else f"{len(flagged)} parameter(s) pressing against the search space:")
report

## 9 · Re-run the full backtest with the picked parameters

In [ ]:
print("picked params:", best.params)
pf_best = bt.run_strategy(strat, panel, best.params, settings=settings)
pd.Series(bt.portfolio_stats(pf_best)).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf_best, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity with picked parameters").show()

## 10 · Verification — per-symbol deep dive

Portfolio means can hide an edge carried by a single name — and this universe
is 20 *correlated* high-volatility names that break out together on
market-wide moves, so breadth here is softer than it looks. First the full
per-symbol breakdown of the optimized backtest; then set `SYMBOL` in the next
cell to any name that looks interesting (the big winner, the one loser, the
one with suspiciously few trades) and re-run it for the same annotated chart
and trade table as §4 — entries, exits, and the gates each break cleared.


In [ ]:
bt.symbol_stats(pf_best).round(4)

In [ ]:
SYMBOL = None   # ← set to any symbol from the table above (e.g. "NVDA"); None = most-traded
sym = SYMBOL or bt.trade_records(pf_best)["symbol"].value_counts().index[0]
inspect_symbol(sym, pf_best, best.params)

## 11 · Does each gate earn its keep? Ablation, cost stress, time-of-day

Three checks between "the optimizer likes it" and "believe it":

1. **Gate ablation** — re-run the picked parameters with one gate disabled at
   a time (the `0`-off-switches built into `signals()`). A gate whose removal
   does *not* degrade risk-adjusted results is not adding selectivity — the
   honest response is to remove or document it, not to ship it silently.
2. **Cost stress** — the headline assumes ~4.5 bps/leg and optimistic stop
   fills. Re-price the pick at 6 bps spread / 5 bps slippage: an edge that
   only exists at best-case friction was never an edge.
3. **Time-of-day** — intraday breakout failure peaks midday, and the
   profile-relative volume gate lowers the *absolute* volume bar exactly
   then. If midday bleeds, a lunch blackout is the v2 knob to add.


In [ ]:
ablations = {
    "picked (all gates)": {},
    "squeeze gate OFF": {"squeeze_thresh": 0.0},
    "volume gate OFF": {"vol_mult": 0.0},
    "trend gate OFF": {"trend_window": 0},
}
rows = {}
for name, off in ablations.items():
    pf_a = bt.run_strategy(strat, panel, {**best.params, **off}, settings=settings)
    rows[name] = bt.portfolio_stats(pf_a)
pd.DataFrame(rows).T.round(4)


In [ ]:
stress = settings.model_copy(deep=True)
stress.costs.spread_bps, stress.costs.slippage_bps = 6.0, 5.0
pf_stress = bt.run_strategy(strat, panel, best.params, settings=stress)
pd.DataFrame({"base costs (§9)": bt.portfolio_stats(pf_best),
              "stressed (6 bps spread / 5 bps slippage)":
                  bt.portfolio_stats(pf_stress)}).round(4)


In [ ]:
trades_best = bt.trade_records(pf_best)
hour = pd.to_datetime(trades_best["entry_ts"], utc=True).dt.tz_convert(panel.tz).dt.hour
trades_best.groupby(hour)["pnl"].agg(
    trades="count", total_pnl="sum",
    win_rate=lambda s: float((s > 0).mean())).round(3)


## 12 · Out-of-sample check — the tail the optimizer never saw

Everything above ran on data up to `data.end`, the **train cutoff**. This
section replays the picked parameters on the untouched tail — the one number
in this notebook the optimizer could not have fit.

The mechanics matter: signals are computed on the **full** history (so the
`trend_window` SMA is warmed up on the day the holdout starts) and only then
are the panel and the signal frames sliced to the holdout rows. Same symbols
as the train panel — re-resolving coverage over the holdout would quietly
change the universe.

**Discipline.** Boundary-widening loops (§8) are judged on TRAIN results
only; this section runs **once per study name**, after the search space is
frozen — every extra "check the tail, tweak, check again" loop quietly fits
the holdout too. The verdict is gated on a minimum OOS trade count (a
selective strategy can produce a tail sample too small to mean anything
either way), and the two next-best admissible front rows are replayed
alongside the pick: a real edge should not flip sign across neighboring
Pareto picks.


In [ ]:
from dataclasses import replace

assert d.get("end"), "set data.end in config.yaml to keep a holdout"
MIN_OOS_TRADES = 50
cutoff = pd.Timestamp(d["end"], tz="UTC")
full = load_panel(engine, panel.symbols, d["bar"], d["start"], None,
                  session=d.get("session", "regular"), exchanges=settings.exchanges)
after = full.close.index > cutoff
assert after.any(), "no bars stored after data.end — backfill the holdout tail first"
fees, slippage = bt.panel_costs(settings.costs, panel)

def oos_portfolio(params):
    """Replay one parameter set on the untouched tail (warm-up sees history)."""
    sig = strat.signals(full, **{**strat.params, **params})
    oos_panel = replace(full, close=full.close[after], open_=full.open_[after],
                        high=full.high[after], low=full.low[after], volume=full.volume[after])
    oos_sig = {k: (v[after] if isinstance(v, pd.DataFrame) else v) for k, v in sig.items()}
    return bt.run(oos_panel, oos_sig, fees=fees, slippage=slippage,
                  init_cash=settings.backtest.init_cash)

pf_oos = oos_portfolio(best.params)
oos = bt.portfolio_stats(pf_oos)
print(f"holdout: {full.close.index[after][0]:%Y-%m-%d} → {full.close.index[-1]:%Y-%m-%d} "
      f"({int(after.sum())} bars, {oos['n_trades']} trades)")
if oos["n_trades"] < MIN_OOS_TRADES:
    print(f"!! only {oos['n_trades']} OOS trades (< {MIN_OOS_TRADES}) — "
          "insufficient sample; do NOT read the metrics below as a verdict")
plots.plot_equity(bt.equity_curve(pf_oos, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — OUT-OF-SAMPLE equity, picked parameters").show()

# Directional consistency: replay the two next-best admissible front rows.
# Params come from the FrozenTrials, NOT the frame rows — iterrows() would
# upcast int params (break_window/horizon/...) to floats and rolling() raises.
by_number = {t.number: t for t in study.best_trials}
table = {"picked": oos}
for trial_no in eligible.sort_values("sharpe", ascending=False)["trial"]:
    if int(trial_no) == best.number or len(table) >= 3:
        continue
    table[f"trial {int(trial_no)}"] = bt.portfolio_stats(
        oos_portfolio(by_number[int(trial_no)].params))
pd.DataFrame({"train (§9)": bt.portfolio_stats(pf_best),
              **{f"holdout · {k}": v for k, v in table.items()}}).round(4)


## Read the result honestly

* §3–§11 are **in-sample**: the optimizer saw all the data they are judged
  on. §12 is the honest number — and even it gets stale after a few re-reads.
  For a fresh view, re-run the pick on a *different universe*.
* **The gates must earn their keep** (§11). If ablating a gate doesn't hurt —
  or the front piles onto a gate's off-edge in §8 — the confluence thesis is
  *disproved* for that gate on this data: remove it or document it, never
  widen the range to chase the off-switch.
* Prefer combos where **`pct_profitable` is high** — but remember these 20
  names are correlated; breadth here substitutes for regime diversity only
  imperfectly (~10 months is a single regime).
* Stop fills are optimistic at the engine level (no slippage on stop legs,
  gap-throughs fill at the stop price) — the §11 cost-stress table is the
  honest floor, not the headline.
* More trials on the same data = more chances to overfit. The Pareto front
  going up does not mean the *strategy* got better.

**Add your own strategy:** copy `strategies/_template/`, write `signals()`, fill
in `config.yaml` — this exact notebook flow works unchanged. See the README's
"How to add a strategy".
